In [ ]:
text_url= "https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:32023R2631"
url1 = "https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:32023R2631"
url2 = "https://eur-lex.europa.eu/legal-content/EN/ALL/?uri=consil:ST_13940_2023_ADD_2_REV_1"

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from graph_builder import GraphBuilder
from collections import deque


In [ ]:

def graph(G):
    """
    Visualize the graph 
    """

    pos = nx.spring_layout(G, seed=50)
    edge_labels = {(u, v): d["relation"] for u, v, d in G.edges(data=True)}
    nx.draw_networkx_nodes(G, pos, node_size=10)
    nx.draw_networkx_labels(G, pos, font_color="white", font_weight="bold")
    nx.draw_networkx_edges(G, pos, edge_color="gray", arrows=True,
                        arrowsize=20, connectionstyle="arc3,rad=0.1")
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)

    plt.title("Directed Graph with Attributes")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

MAX_DEPTH = 5
MAX_WORKERS = 10

def fetch_graph_and_links(url):
    """Single thread worker: builds the subgraph AND extracts links in one shot."""
    try:
        g_init = GraphBuilder(url)
        
        sub_graph = g_init.create_graph()  # build this page's subgraph
        links = set(g_init.find_links())   # reuse the SAME GraphBuilder instance — no double fetch
        
        return url, sub_graph, links
    except Exception as e:
        print(f"  [Error] {url}: {e}")
        return url, None, set()


def collect_and_build_graph(start_url):

    G = nx.DiGraph()          # the combined graph, grown as we crawl
    visited = set()           # keep track of visted url
    seen = {start_url}        # keep track of quequed ones
    all_urls = []
    current_layer = [(start_url, 0)]

    while current_layer:
        next_layer = []

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {
                executor.submit(fetch_graph_and_links, url): (url, layer)
                for url, layer in current_layer
                if url not in visited
            }

            for future in as_completed(futures):
                url, layer = futures[future]
                fetched_url, sub_graph, links = future.result()

                if fetched_url in visited:
                    continue
                visited.add(fetched_url)
                all_urls.append(fetched_url)
                print(f"[Layer {layer}] Visited: {fetched_url} — {len(links)} links found")

                # Merge this page's subgraph into the main graph
                if sub_graph is not None and len(sub_graph.nodes) > 0:
                    G = nx.compose(G, sub_graph)

                if layer >= MAX_DEPTH:
                    continue

                new_links = links - seen
                seen.update(new_links)
                for link in new_links:
                    next_layer.append((link, layer + 1))

        current_layer = next_layer

    print(f"\nTotal URLs collected: {len(all_urls)}")
    return G, all_urls   # return both so you can use either

In [ ]:
agraph = build_full_graph(url1)

In [ ]:
graph(agraph)